# Step 1: Prepare & curate the SQuAD dataset (run locally)

In [ ]:
from pathlib import Path
import pandas as pd
from datasets import load_dataset

OUTPUT_DIR = Path("prepared_data")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load raw SQuAD (v1.1 train split)

In [ ]:
df = load_dataset("squad", split="train").to_pandas()

## 2. Curated topic list

`split` marks each title as train or test; a title appearing in **both** is a *seen* test
topic. `category` is metadata only - it is not used in the selection. Edit this table to
curate the dataset.

In [3]:
data = [
    ['American_Idol', 'train', 'music'],
    ['Adult_contemporary_music', 'train', 'music'],
    ['Classical_music', 'train', 'music'],
    ['House_music', 'train', 'music'],
    ['Hard_rock', 'train', 'music'],
    ['Queen_(band)', 'train', 'music'],
    ['A_cappella', 'train', 'music'],
    ['Sino-Tibetan_relations_during_the_Ming_dynasty', 'train', 'history'],
    ['Dutch_Republic', 'train', 'history'],
    ['Middle_Ages', 'train', 'history'],
    ['Late_Middle_Ages', 'train', 'history'],
    ['Franco-Prussian_War', 'train', 'history'],
    ['Age_of_Enlightenment', 'train', 'history'],
    ['Hellenistic_period', 'train', 'history'],
    ['British_Empire', 'train', 'history'],
    ['Buddhism', 'train', 'philosophy'],
    ['Sumer', 'train', 'history'],
    ['Political_philosophy', 'train', 'philosophy'],
    ['Philosophy_of_space_and_time', 'train', 'philosophy'],
    ['Empiricism', 'train', 'philosophy'],
    ['Idealism', 'train', 'philosophy'],
    ['Humanism', 'train', 'philosophy'],
    ['Arena_Football_League', 'train', 'sports'],
    ['England_national_football_team', 'train', 'sports'],
    ['Gymnastics', 'train', 'sports'],
    ['FC_Barcelona', 'train', 'sports'],
    ['Everton_F.C.', 'train', 'sports'],
    ['Arsenal_F.C.', 'train', 'sports'],
    ['Canadian_football', 'train', 'sports'],
    ['Premier_League', 'train', 'sports'],
    ['Association_football', 'train', 'sports'],
    ['House_music', 'test', 'music'],
    ['Hard_rock', 'test', 'music'],
    ['Queen_(band)', 'test', 'music'],
    ['Gramophone_record', 'test', 'music'],
    ['Mandolin', 'test', 'music'],
    ['Post-punk', 'test', 'music'],
    ['Myocardial_infarction', 'test', 'diseases'],
    ['Tuberculosis', 'test', 'diseases'],
    ['Asthma', 'test', 'diseases'],
    ['Diarrhea', 'test', 'diseases'],
    ['Pain', 'test', 'diseases'],
    ['Infection', 'test', 'diseases'],
    ['Financial_crisis_of_2007%E2%80%9308', 'test', 'economics'],
    ['United_States_dollar', 'test', 'economics'],
    ['Economy_of_Greece', 'test', 'economics'],
    ['Age_of_Enlightenment', 'test', 'history'],
    ['Umayyad_Caliphate', 'test', 'history'],
    ['Hellenistic_period', 'test', 'history'],
    ['British_Empire', 'test', 'history'],
    ['Roman_Republic', 'test', 'history'],
    ["Kievan_Rus%27", 'test', 'history'],
    ['Buddhism', 'test', 'philosophy'],
    ['Materialism', 'test', 'philosophy'],
    ['Canon_law', 'test', 'philosophy'],
    ['Freemasonry', 'test', 'philosophy'],
    ['Political_philosophy', 'test', 'philosophy'],
    ['Humanism', 'test', 'philosophy'],
    ['FC_Barcelona', 'test', 'sports'],
    ['Everton_F.C.', 'test', 'sports'],
    ['Arsenal_F.C.', 'test', 'sports'],
    ['Chicago_Cubs', 'test', 'sports'],
    ['FA_Cup', 'test', 'sports'],
    ['Exhibition_game', 'test', 'sports'],
]
title_df = pd.DataFrame(data, columns=['title', 'split', 'category'])

# Sanity-check: every curated title exists in SQuAD
missing = set(title_df.title) - set(df.title)
assert not missing, f"titles not found in SQuAD: {missing}"
print(f"{title_df.shape[0]} rows, {title_df.title.nunique()} unique titles")

64 rows, 52 unique titles


## 3. Select rows

- **train** = 90% of questions per train title
- **test**  = the held-out ~10% of *seen* titles + 20% of questions per *unseen* title

In [4]:
train_titles = title_df.loc[title_df.split == 'train', 'title']
test_titles = title_df.loc[title_df.split == 'test', 'title']

train_df = df[df.title.isin(train_titles)].groupby('title').sample(frac=0.9, random_state=42)

seen = set(train_df.title) & set(test_titles)          # titles in both -> seen test topics
unseen = set(test_titles) - seen                        # test-only titles
dev_seen = df[df.title.isin(seen) & ~df.id.isin(train_df.id)]
dev_unseen = df[df.title.isin(unseen)].groupby('title').sample(frac=0.2, random_state=42)
test_df = pd.concat([dev_seen, dev_unseen])

## 4. Derive answer + prompt columns

Both frames get `answer_text` **and** `answer_start` (the original notebook set only one on
each, leaving the test set without a gold answer).

In [5]:
def create_prompt(row):
    return (f"Look at the context given below and answer the question."
            f"Answer in as few words as possible.\nContext: {row['context']}\nQuestion: {row['question']}")

for d in (train_df, test_df):
    d['answer_text'] = d['answers'].apply(lambda a: a['text'][0])
    d['answer_start'] = d['answers'].apply(lambda a: int(a['answer_start'][0]))
    d['prompt'] = d.apply(create_prompt, axis=1)

## 5. Curate: inspect before saving


In [6]:
for name, d in [('train', train_df), ('test', test_df)]:
    print(f"{name:5s}: {len(d):5d} rows, {d.title.nunique():2d} titles")
print("\nseen test topics  :", sorted(seen))
print("unseen test topics:", sorted(unseen))
print("\nrows per split/category:")
print(title_df.groupby(['split', 'category']).size())

train:  6620 rows, 31 titles
test :  1056 rows, 33 titles

seen test topics  : ['Age_of_Enlightenment', 'Arsenal_F.C.', 'British_Empire', 'Buddhism', 'Everton_F.C.', 'FC_Barcelona', 'Hard_rock', 'Hellenistic_period', 'House_music', 'Humanism', 'Political_philosophy', 'Queen_(band)']
unseen test topics: ['Asthma', 'Canon_law', 'Chicago_Cubs', 'Diarrhea', 'Economy_of_Greece', 'Exhibition_game', 'FA_Cup', 'Financial_crisis_of_2007%E2%80%9308', 'Freemasonry', 'Gramophone_record', 'Infection', 'Kievan_Rus%27', 'Mandolin', 'Materialism', 'Myocardial_infarction', 'Pain', 'Post-punk', 'Roman_Republic', 'Tuberculosis', 'Umayyad_Caliphate', 'United_States_dollar']

rows per split/category:
split  category  
test   diseases      6
       economics     3
       history       6
       music         6
       philosophy    6
       sports        6
train  history       9
       music         7
       philosophy    6
       sports        9
dtype: int64


## 6. Save locally

In [ ]:
COLS = ['id', 'title', 'context', 'question', 'answer_text', 'answer_start', 'prompt']
train_df[COLS].reset_index(drop=True).to_parquet(OUTPUT_DIR / "train.parquet", index=False)
test_df[COLS].reset_index(drop=True).to_parquet(OUTPUT_DIR / "test.parquet", index=False)
print("wrote", (OUTPUT_DIR / "train.parquet").resolve())
print("wrote", (OUTPUT_DIR / "test.parquet").resolve())

## 7. Upload to AWS

Copy the two files into S3 at the pipeline's `raw/{split}.parquet` location. From a SageMaker
notebook (with `aws_config` importable):

```python
import pandas as pd, aws_config as cfg
for s in ("train", "test"):
    pd.read_parquet(f"{s}.parquet").to_parquet(cfg.raw_uri(s), index=False)
```

or with the CLI (find the bucket via `sagemaker.Session().default_bucket()`):

```bash
aws s3 cp train.parquet s3://<bucket>/confidence-llm/raw/train.parquet
aws s3 cp test.parquet  s3://<bucket>/confidence-llm/raw/test.parquet
```

> **Prompt note:** `02_run_llm.py` rebuilds this exact prompt from the `context` and
> `question` columns and wraps it in the Qwen chat template, so the stored `prompt` column is
> a reference copy rather than an input the pipeline reads. The region masks in
> `attention_features_v2.py` locate the `Context :` / `Question :` markers dynamically and
> assume this prompt wording.